# Gradient Descent in 2D

In this notebook we study the gradient descent method for minimizing functions of two variables.  
The presentation starts with a quadratic objective, where the geometry is simple and the behaviour of the method can be understood analytically. We then extend the discussion to non-convex examples.

The notebook is organized as follows:
- definition of the objective function and its gradient;
- computation of the minimizer with a standard numerical solver;
- implementation of steepest descent with fixed step size;
- analysis of the effect of the learning rate;
- implementation of backtracking line search with the Armijo condition;
- application to the Ackley and Griewank functions.

Throughout the notebook, we use the notation $\mathbf{x} = (x_1,x_2)^T$.

## Definitions

We begin with a quadratic objective function. This example is useful because:
- the minimizer can be identified exactly;
- the gradient has a simple closed form;
- the effect of the step size can be visualized clearly on the contour plot.

Since the function is quadratic and strictly convex, it has a unique global minimizer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


We want to minimize the function

\begin{equation}
f(\mathbf{x}) = 0.5 \left( x_1 - 4.5 \right)^2 + 2.5 \left( x_2 - 2.3 \right)^2.
\end{equation}

This function is the sum of two squared terms centered at $x_1 = 4.5$ and $x_2 = 2.3$.  
Therefore, its unique minimizer is expected at the point where both squared terms vanish simultaneously.

The gradient of this function is

\begin{equation}
\nabla f(\mathbf{x}) =
\begin{bmatrix}
\displaystyle \frac{\partial f}{\partial x_1} \\
\displaystyle \frac{\partial f}{\partial x_2}
\end{bmatrix}
=
\begin{bmatrix}
x_1 - 4.5 \\
5\left(x_2 - 2.3\right)
\end{bmatrix}.
\end{equation}

The gradient points in the direction of steepest increase of $f$.  
Consequently, the direction $-\nabla f(\mathbf{x})$ is the direction of steepest local decrease.

Define two functions:

- one that, given a point $\mathbf{x}$, computes the objective value $f(\mathbf{x})$;
- one that, given a point $\mathbf{x}$, computes the gradient $\nabla f(\mathbf{x})$.

This separation is important in optimization, since many algorithms need both the function value and first-order derivative information.

In [ ]:
# write your code here 
def f(x):
    '''Objective function'''
    return
    
def df(x):
    '''Gradient of the objective function'''
    return  

Compute the minimum of this function using the `scipy` library [SciPy](https://scipy.org/) and, in particular, the routine [`scipy.optimize.minimize`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html).

Use:
- `np.zeros(2)` as the initial point `x0`;
- `method='trust-constr'`;
- `jac=df` to provide the gradient explicitly.

The numerical result should agree with the analytical minimizer of the quadratic function.

```python
# Expected output:
# array([4.5, 2.3])
```

In [ ]:
from scipy.optimize import minimize

result = minimize(
    f, np.zeros(2), method='trust-constr', jac=df)

result.x

Plot the objective function and the minimizer obtained with `scipy`.

The surface plot and the contour plot should confirm that the minimum is attained at the same point computed above.  
This is a useful consistency check before implementing our own optimization routine.

In [ ]:
# Generate mesh grid for the function visualization
x_vals = np.linspace(-10, 10, 20)
y_vals = np.linspace(-10, 10, 20)
X_grid, Y_grid = np.meshgrid(x_vals, y_vals)
Z_values = f(np.array([X_grid, Y_grid]))

# Extract minimizer coordinates
opt_x0, opt_x1 = np.meshgrid(result.x[0], result.x[1])
opt_z = f(np.stack([opt_x0, opt_x1]))

# Initialize figure
fig = plt.figure(figsize=(15, 20))

# First 3D contour plot
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
ax1.contour3D(X_grid, Y_grid, Z_values, levels=60, cmap='viridis')
ax1.scatter(opt_x0, opt_x1, opt_z, color='red', s=80, label='Minimizer')
ax1.set_xlabel('$x_0$')
ax1.set_ylabel('$x_1$')
ax1.set_zlabel('$f(x)$')
ax1.view_init(elev=40, azim=20)
ax1.legend()

# Second 3D contour plot (Top-down view)
ax2 = fig.add_subplot(1, 2, 2, projection='3d')
ax2.contour3D(X_grid, Y_grid, Z_values, levels=60, cmap='viridis')
ax2.scatter(opt_x0, opt_x1, opt_z, color='red', s=80, label='Minimizer')
ax2.set_xlabel('$x_0$')
ax2.set_ylabel('$x_1$')
ax2.set_zlabel('$f(x)$')
ax2.axes.zaxis.set_ticklabels([])
ax2.view_init(elev=90, azim=-90)  # Top-down view
ax2.legend()

plt.show()


## Steepest Descent Method

We now implement the steepest descent method, also known in many machine learning contexts as gradient descent.

At every iteration, the method chooses a descent direction and a step length.  
For steepest descent, the search direction is the negative gradient.

Given an initial point $\mathbf{x}_0$, the method uses the update rule

\begin{equation}
\mathbf{x}_{k+1} = \mathbf{x}_k + \alpha_k \mathbf{p}_k,
\end{equation}

where:
- $\mathbf{p}_k$ is the search direction;
- $\alpha_k > 0$ is the step length.

For the steepest descent method, the search direction is chosen as

\begin{equation}
\mathbf{p}_k = -\nabla f(\mathbf{x}_k).
\end{equation}

Substituting this choice into the update gives

\begin{equation}
\mathbf{x}_{k+1} = \mathbf{x}_k - \alpha_k \nabla f(\mathbf{x}_k).
\end{equation}

Hence, the method moves against the gradient, with a displacement scaled by $\alpha_k$.

**Algorithm**

A basic implementation of the steepest descent method proceeds as follows:

1. Choose an initial point $\mathbf{x}_0$.
2. Fix a maximum number of iterations $M$.
3. Fix a tolerance `tol` for the stopping criterion.
4. For $k=0,1,2,\dots$:
   - compute the gradient $\nabla f(\mathbf{x}_k)$;
   - choose a step length $\alpha_k$;
   - update
     \begin{equation}
     \mathbf{x}_{k+1} = \mathbf{x}_k - \alpha_k \nabla f(\mathbf{x}_k);
     \end{equation}
   - stop if $\|\nabla f(\mathbf{x}_k)\| < \mathrm{tol}$ or if $k$ reaches $M$.

For a differentiable function, the condition $\|\nabla f(\mathbf{x}_k)\| \approx 0$ indicates that the iterate is close to a stationary point.

In [ ]:
def steepest_descent(gradient, x0=np.zeros(2), alpha=0.01, max_iter=10000, tolerance=1e-10): 
    '''
    Steepest descent with a fixed step size.

    Args:
      - gradient: Function computing the gradient of the objective.
      - x0: Initial guess for (x_0, x_1) (default: zeros) <numpy.ndarray>.
      - alpha: Step size parameter (default: 0.01).
      - max_iter: Maximum number of iterations (default: 10,000).
      - tolerance: Convergence criterion based on gradient norm (default: 1e-10).

    Returns:
      - results: <numpy.ndarray> of shape (n_iter, 2) with (x_0, x_1) at each step.
      - num_steps: <int> Total number of iterations performed.
    '''

    # Initialize the array to store iterations
    results = np.array([])

    # Compute initial gradient
    grad_val = gradient(x0)

    # Initialize step counter
    num_iterations = 0

    # Set initial point
    x = x0
    results = np.append(results, x, axis=0)

    # Iterate until convergence or max iterations reached
    # We stop when the gradient is close to zero
    while any(abs(grad_val) > tolerance) and num_iterations < max_iter:

        # Move in the direction of the negative gradient x_{k+1} = x_k - alpha * grad f(x_k)

        
        # Store new point (use append if you want)

        
        # Compute new gradient

        
        # Increment step counter

        
    # Reshape results for correct output format
    # POSSIBLE OUTPUT, USE WHAT YOU PREFER
    return results.reshape(-1, 2), num_iterations


Call this function to minimize $f(\mathbf{x})$.

The output should show an approximation of the minimizer together with the number of iterations required for convergence.

```python
# Expected output:
# - Optimal solution: [4.5 2.3]
# - Total iterations: 72
```

In [ ]:
# Perform steepest descent optimization
trajectory, num_iterations = steepest_descent(
    df, x0=np.array([-9, -9]), alpha=0.30
)

# Extract the final minimizer
optimal_point = trajectory[-1].round(1)

# Display the results
print('- Optimal solution:', optimal_point)
print('- Total iterations:', num_iterations)


## Visualization
The next function plots the optimization trajectory over the graph of the objective function.

This visualization is useful to interpret the dynamics of the method:
- on the surface plot, we see how the iterates move in three dimensions;
- on the contour plot, we see the same path projected onto the $(x_1,x_2)$ plane.

In [ ]:
def plot_optimization_trajectory(trajectory, num_iterations, objective_function):
    """
    Plots the optimization trajectory on a 3D surface plot with the objective function contours.
    
    Args:
        - trajectory: <numpy.ndarray> of size (n_iter, 2) with x_0 and x_1 values at each iteration
        - num_iterations: <int> the total number of iterations performed
        - objective_function: function to calculate the objective value, takes a 2D array as input
    """
    # Create a meshgrid for X and Y values for the plot
    X, Y = np.meshgrid(np.linspace(-10, 10, 50), np.linspace(-10, 10, 50))

    # Calculate Z values (the objective function values)
    Z = objective_function(np.array([X, Y]))

    # Extract coordinates of iterates
    X_vals, Y_vals = trajectory[:, 0], trajectory[:, 1]
    Z_vals = objective_function(np.array([X_vals, Y_vals]))

    # Create figure
    fig = plt.figure(figsize=(20, 20))

    # First 3D plot
    ax1 = fig.add_subplot(1, 2, 1, projection='3d')
    ax1.contour3D(X, Y, Z, levels=60, cmap='viridis')
    ax1.plot(X_vals, Y_vals, Z_vals, color='red', linewidth=3)
    ax1.scatter(trajectory[-1, 0], trajectory[-1, 1], objective_function(trajectory[-1]), marker='o', color='red', s=100)
    ax1.set_xlabel('$x_0$')
    ax1.set_ylabel('$x_1$')
    ax1.set_zlabel('$f(x)$')
    ax1.view_init(elev=20, azim=20)

    # Second 3D plot
    ax2 = fig.add_subplot(1, 2, 2, projection='3d')
    ax2.contour3D(X, Y, Z, levels=60, cmap='viridis')
    ax2.plot(X_vals, Y_vals, Z_vals, color='red', linewidth=3)
    ax2.scatter(trajectory[-1, 0], trajectory[-1, 1], objective_function(trajectory[-1]), marker='o', color='red', s=100)
    ax2.set_xlabel('$x_0$')
    ax2.set_ylabel('$x_1$')
    ax2.set_zlabel('$f(x)$')
    ax2.axes.zaxis.set_ticklabels([])
    ax2.view_init(elev=90, azim=-90)

    plt.show()


Plot the trajectory by calling the function below.

For this quadratic example, the path should move toward the unique minimizer.  
Because the level sets are elongated ellipses, the iterates may exhibit a zig-zag pattern before converging.

In [ ]:
# Call the function to plot the results
plot_optimization_trajectory(trajectory, num_iterations, f)

## Analysis of the Step Size

Try the step sizes (learning rates in other contexts) $[0.01, 0.25, 0.3, 0.35, 0.4]$ and analyze what happens.

This experiment illustrates a central issue in gradient methods:
- if the step size is too small, convergence is slow;
- if the step size is well chosen, convergence is efficient;
- if the step size is too large, the iterates may oscillate or diverge.

For a quadratic function, these effects are especially easy to observe.

In [ ]:
# Define step sizes to test
step_sizes = [0.01, 0.25, 0.3, 0.35, 0.4]


## PLOT THE LOSS FUNCTIONS

# Arrays to store optimization paths
x_trajectories, y_trajectories, z_trajectories = [], [], []

# Set up figure for function value plots
fig, axes = plt.subplots(len(step_sizes), figsize=(8, 9))
fig.suptitle('Function Value $f(x)$ Across Iterations for Different Step Sizes')

# Loop through different step sizes
for idx, step in enumerate(step_sizes):

    # Run steepest descent for the current step size
    path, iterations = steepest_descent(
        df, x0=np.array([-5, -5]), alpha=step, max_iter=3000
    )

    # Display results
    print(f'Testing alpha = {step}')
    print(f'  - Optimal solution: {path[-1].round(1)}')
    print(f'  - Total iterations: {iterations}')

    # Store optimization trajectory for later visualization
    x_trajectories.append(path[:, 0])
    y_trajectories.append(path[:, 1])
    z_trajectories.append(f(np.array([path[:, 0], path[:, 1]])))

    # Plot function values across iterations
    axes[idx].plot([f(p) for p in path], label=f'α = {step}')
    axes[idx].axhline(y=0, color='r', alpha=0.7, linestyle='dashed')
    axes[idx].set_xlabel('Iterations')
    axes[idx].set_ylabel('$f(x)$')
    axes[idx].set_ylim([-10, 200])
    axes[idx].legend(loc='upper right')


## PLOT THE LOSS PATHS


# Prepare the objective function between -10 and 10
X, Y = np.meshgrid(np.linspace(-10, 10, 20), np.linspace(-10, 10, 20))
Z = f(np.array([X, Y]))

# Find the minimum point using steepest descent
optimal_point = trajectory[-1]  # Last point from the descent
min_x0, min_x1 = optimal_point  # Extract x0 and x1
min_z = f(optimal_point)  # Compute function value at the minimum

# Loop through each step size to generate plots
fig = plt.figure(figsize=(25, 60))

for idx, step in enumerate(step_sizes):

    # First 3D plot
    ax1 = fig.add_subplot(5, 2, (idx * 2) + 1, projection='3d')
    ax1.contour3D(X, Y, Z, levels=60, cmap='viridis')
    ax1.plot(x_trajectories[idx], y_trajectories[idx], z_trajectories[idx], 
             color='red', linewidth=3, label=f'α = {step}')
    ax1.scatter(min_x0, min_x1, min_z, marker='o', color='red', s=100)
    ax1.set_xlabel('$x_0$')
    ax1.set_ylabel('$x_1$')
    ax1.set_zlabel('$f(x)$')
    ax1.view_init(20, 20)
    ax1.legend(prop={'size': 15})

    # Second 3D plot with a different perspective
    ax2 = fig.add_subplot(5, 2, (idx * 2) + 2, projection='3d')
    ax2.contour3D(X, Y, Z, levels=60, cmap='viridis')
    ax2.plot(x_trajectories[idx], y_trajectories[idx], z_trajectories[idx], 
             color='red', linewidth=3, label=f'α = {step}')
    ax2.scatter(min_x0, min_x1, min_z, marker='o', color='red', s=100)
    ax2.set_xlabel('$x_0$')
    ax2.set_ylabel('$x_1$')
    ax2.set_zlabel('$f(x)$')
    ax2.axes.zaxis.set_ticklabels([])
    ax2.view_init(90, -90)
    ax2.legend(prop={'size': 15})


## Armijo Line Search

Implement the **Armijo condition** to select the step length.

\begin{equation}
f(\mathbf{x}_{k+1}) \leq f(\mathbf{x}_k) + c_1 \alpha_k \nabla f(\mathbf{x}_k)^T \mathbf{p}_k,
\qquad 0 < c_1 < 1.
\end{equation}

Since $\mathbf{x}_{k+1} = \mathbf{x}_k + \alpha_k \mathbf{p}_k$, this condition requires a sufficient decrease of the objective function along the chosen search direction.

For gradient descent we take

\begin{equation}
\mathbf{p}_k = -\nabla f(\mathbf{x}_k).
\end{equation}

Use the following backtracking strategy:
1. start from an initial trial value $\alpha_0 = 1$;
2. test whether the Armijo condition is satisfied;
3. if it is not satisfied, replace $\alpha$ by a smaller value (here by repeated halving);
4. once the condition is satisfied, accept that step length.

This procedure is called **backtracking line search**.


- The Armijo condition guarantees **global convergence** under standard assumptions (e.g., Lipschitz continuous gradient).
- The parameter $c_1$ controls how much decrease is required (typically $10^{-4}$).
- The parameter $\rho$ controls how aggressively the step size is reduced (typically $0.5$).
- Compared to a fixed step size, this approach is more **robust** and adapts to the local geometry of $f$.

In [ ]:
def linesearcharmijo(fun, xk, pk, gfk, alpha0, rho=0.5, c1=1e-4):
    '''
    Minimize over alpha, the function f(xₖ + αpₖ).
    α > 0 is assumed to be a descent direction.
    
    Parameters
    --------------------
    fun: callable
        Function to be minimized.
    xk : array
        Current point.
    pk : array
        Search direction.
    gfk : array
        Gradient of f at point xk.
    alpha0 : scalar
        Value of alpha at the start of the optimization.
    rho : float, optional
        Value of alpha shrinkage factor.
    c1 : float, optional
        Value to control stopping criterion.
    
    Returns
    --------------------
    alpha : scalar
        Value of alpha at the end of the optimization.
    phi : float
        Value of f at the new point x_{k+1}.
    '''
    
    # write your code here 
    derf0 = np.dot(gfk, pk)
    f_a0 = fun(xk + alpha0*pk)
    f_x=fun(xk)

    # while the condition is not satisfied
    while not  :
 
    
    return alpha0

Include the line search in the gradient descent function.

The goal is to replace the fixed step size by an adaptive choice of $\alpha_k$, computed at each iteration through the Armijo backtracking procedure.

In [ ]:
def steepest_descent_ls(fun, gradient, x0=np.zeros(2), alpha=1, max_iter=10000, tolerance=1e-10):  
    '''
    Steepest descent optimization using an adaptive step size (Armijo condition).

    Args:
      - f : Callable Function to be minimized.
      - gradient: Callable function computing the gradient of the objective function.
      - x0: Initial guess for (x_0, x_1) (default: zeros) <numpy.ndarray>.
      - max_iter: Maximum number of iterations allowed (default: 10,000).
      - tolerance: Convergence criterion based on gradient norm (default: 1e-10).

    Returns:
      - results: <numpy.ndarray> of shape (n_iter, 2) storing (x_0, x_1) at each step.
      - num_iterations: <int> Total number of iterations performed.
    '''

    # Initialize storage for the results
    results = np.array([])

    # Initialize iteration counter
    num_iterations = 0
    
    # Set the initial point and the initial gradient
    xk = x0
    grad_val_k = gradient(x0)
    
    results = np.append(results, xk, axis=0)

    # Iterate until convergence or reaching the iteration limit
    while any(abs(grad_val_k) > tolerance) and num_iterations < max_iter:

        # determine descending direction
        pk = -grad_val_k
        
        # Compute step size using Armijo condition (adaptive step size)
        step_size = linesearcharmijo(fun, xk, pk, grad_val_k, alpha0=alpha, rho=0.5, c1=1e-4)

        # Update current position by moving along the negative gradient
        xk = xk + step_size * pk

        # Store the new point in the results
        results = np.append(results, xk, axis=0)

        # Compute the new gradient
        grad_val_k = gradient(xk)

        # Increment iteration counter
        num_iterations += 1

    # Found minimizer
#    minimizer = results[-1].round(1)
    f_val=fun(xk)

    # Print results
    print('- Function minima:', f_val)
    print('- Final point:', xk)
    print('- N° steps:  {}'.format(num_iterations))

    # Reshape and return results
    return f_val, results.reshape(-1, 2), num_iterations


### Example I: 
Compute the minimum of $f(\mathbf{x})$ using steepest descent with line search.

Compare the result with:
- the exact minimizer of the quadratic function;
- the approximation previously obtained with a fixed step size;
- the solution returned by `scipy.optimize.minimize`.

In [ ]:
# Steepest descent with fixed step size
trajectory, num_iterations = steepest_descent(
    df, x0=np.array([-9.0, -9.0]), alpha=0.01
)
# Extract the final minimizer
optimal_point = trajectory[-1].round(1)

# Display the results
print('- Optimal solution:', optimal_point)
print('- Total iterations:', num_iterations)

In [ ]:
# Steepest descent with line search
minima, points, iters = steepest_descent_ls(
  f, df, x0 = np.array([-9, -9]), tolerance=1e-6)


### Example II: Ill-Conditioned Quadratic Function

Consider the quadratic function
\begin{equation}
f(x,y) = \frac{1}{2}\left(x^2 + 100\,y^2\right).
\end{equation}

Its gradient is given by
\begin{equation}
\nabla f(x,y) =
\begin{bmatrix}
x \\
100y
\end{bmatrix}.
\end{equation}

---

### Properties

- The function is **convex** and has a unique minimizer at
\begin{equation}
(x^\star, y^\star) = (0,0).
\end{equation}

- The Hessian matrix is
\begin{equation}
\nabla^2 f(x,y) =
\begin{bmatrix}
1 & 0 \\
0 & 100
\end{bmatrix}.
\end{equation}

- The eigenvalues of the Hessian are
\begin{equation}
\lambda_1 = 1, \quad \lambda_2 = 100,
\end{equation}
so the condition number is
\begin{equation}
\kappa = \frac{\lambda_{\max}}{\lambda_{\min}} = 100.
\end{equation}

---

### Interpretation

- The level sets of $f$ are **elongated ellipses**, reflecting strong anisotropy.
- The curvature in the $y$-direction is much larger than in the $x$-direction.
- As a consequence, steepest descent methods tend to exhibit **zig-zag behavior**.

---

### Relevance for Line Search

This function is a standard test case to illustrate the role of adaptive step sizes:

- A fixed step size must be chosen carefully to ensure convergence.
- The Armijo backtracking rule automatically reduces the step size when needed.
- This leads to a more stable and robust behavior, especially in ill-conditioned problems.

In [ ]:
def f_ill_cond(x):
    """
    Ill-conditioned quadratic function:
        f(x, y) = 0.5 * (x^2 + 100 y^2)
    """
    return 0.5 * (x[0]**2 + 100 * x[1]**2)


def grad_ill_cond(x):
    """
    Gradient of f:
        ∇f(x, y) = [x, 100y]
    """
    return np.array([
        x[0],
        100 * x[1]
    ])

In [ ]:
# Steepest descent with fixed step size (THIS WILL FAIL because it diverges for this size)
trajectory, num_iterations = steepest_descent(
    grad_ill_cond, x0=np.array([-9.0, -9.0]), alpha=0.30
)
print("Number of stored points:", trajectory.shape[0])

In [ ]:
# Steepest descent with fixed step size
trajectory, num_iterations = steepest_descent(
    grad_ill_cond, x0=np.array([-9.0, -9.0]), alpha=0.01
)
# Extract the final minimizer
optimal_point = trajectory[-1].round(1)

# Display the results
print('- Optimal solution:', optimal_point)
print('- Total iterations:', num_iterations)


In [ ]:
# Steepest descent with Armijo line search
minima, points, iters = steepest_descent_ls(
    f_ill_cond, grad_ill_cond, x0=np.array([-9.0, -9.0]), tolerance=1e-6
)


## EXTRA (individual work)

### Ackley function:

Repeat the process for the **Ackley function**:

\begin{equation}
f(x) = -20 \exp\left(-0.2 \sqrt{\frac{x_1^2 + x_2^2}{2}}\right)
- \exp\left(\frac{1}{2}\left(\cos(2 \pi x_1) + \cos(2 \pi x_2)\right)\right)
+ \exp(1) + 20.
\end{equation}

This is a standard non-convex test function in optimization.  
It has many local irregularities and a global minimum near the origin, so it provides a more demanding test than the quadratic example.

In [ ]:
def ackley(x):
    """
    Ackley function for a 2D input.

    Args:
        x (numpy.ndarray): Input vector of shape (2,)

    Returns:
        float: Function value at x.
    """
    term1 = -20 * np.exp(-0.2 * np.sqrt(0.5 * (x[0]**2 + x[1]**2)))
    term2 = -np.exp(0.5 * (np.cos(2 * np.pi * x[0]) + np.cos(2 * np.pi * x[1])))
    return term1 + term2 + np.e + 20

def ackley_gradient(x):
    """
    Gradient of the Ackley function for a 2D input.

    Args:
        x (numpy.ndarray): Input vector of shape (2,)

    Returns:
        numpy.ndarray: Gradient vector of shape (2,)
    """
    term1_x1 = (20 * x[0]) / np.sqrt(0.5 * (x[0]**2 + x[1]**2)) * np.exp(-0.2 * np.sqrt(0.5 * (x[0]**2 + x[1]**2)))
    term1_x2 = (20 * x[1]) / np.sqrt(0.5 * (x[0]**2 + x[1]**2)) * np.exp(-0.2 * np.sqrt(0.5 * (x[0]**2 + x[1]**2)))
    
    term2_x1 = 2 * np.pi * np.sin(2 * np.pi * x[0]) * np.exp(0.5 * (np.cos(2 * np.pi * x[0]) + np.cos(2 * np.pi * x[1])))
    term2_x2 = 2 * np.pi * np.sin(2 * np.pi * x[1]) * np.exp(0.5 * (np.cos(2 * np.pi * x[0]) + np.cos(2 * np.pi * x[1])))

    grad_x1 = term1_x1 + term2_x1
    grad_x2 = term1_x2 + term2_x2
    
    return np.array([grad_x1, grad_x2])

Plot the shape of the Ackley function by evaluating the following code.

Observe that the function landscape is no longer a simple bowl-shaped surface.  
This makes the behaviour of gradient-based methods more sensitive to the starting point and to the step-size rule.

In [ ]:
from itertools import product  # Import the product function from itertools

# Define the Griewank function (make sure it's defined or replace with the correct one)
#def Griewank_plot(x):
    # Example implementation of Griewank function
#    return 1 + (1/4000) * (x[0]**2 + x[1]**2) - np.cos(x[0]) * np.cos(x[1] / np.sqrt(2))

# Generate grid
x = np.arange(-5, 5, 0.025)
y = np.arange(-5, 5, 0.025)
X, Y = np.meshgrid(x, y)
Z = np.zeros(X.shape)

# Iterate through the grid to compute Z values using the Griewank function
mesh_size = range(len(X))
for i, j in product(mesh_size, mesh_size):
    x_coor = X[i][j]
    y_coor = Y[i][j]
    Z[i][j] = ackley(np.array([x_coor, y_coor]))

# Plot the surface
fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection='3d')  # Use add_subplot with 'projection' argument
ax.set_title('2D Ackley Function')
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_zlabel('$f(x_1, x_2)$')
ax.plot_surface(X, Y, Z, cmap='viridis')
plt.tight_layout()
plt.show()

Use the following plotting function to represent the optimization results in the next examples.

The plotting range is wider, which is convenient for non-convex functions with more complex geometry.

In [ ]:
def plot_optimization_trajectory_wider(trajectory, num_iterations, objective_function):
    """
    Plots the optimization trajectory on a 3D surface plot with the objective function contours.
    
    Args:
        - trajectory: <numpy.ndarray> of size (n_iter, 2) with x_0 and x_1 values at each iteration
        - num_iterations: <int> the total number of iterations performed
        - objective_function: function to calculate the objective value, takes a 2D array as input
    """
    # Create a meshgrid for X and Y values for the plot
    X, Y = np.meshgrid(np.linspace(-10, 10, 50), np.linspace(-10, 10, 50))

    # Calculate Z values (the objective function values)
    Z = objective_function(np.array([X, Y]))

    # Extract coordinates of iterates
    X_vals, Y_vals = trajectory[:, 0], trajectory[:, 1]
    Z_vals = objective_function(np.array([X_vals, Y_vals]))

    # Create figure
    fig = plt.figure(figsize=(20, 20))

    # First 3D plot (with surface and trajectory)
    ax1 = fig.add_subplot(1, 2, 1, projection='3d')
    ax1.contour3D(X, Y, Z, levels=60, cmap='viridis')
    ax1.plot(X_vals, Y_vals, Z_vals, color='red', linewidth=3)
    ax1.scatter(trajectory[-1, 0], trajectory[-1, 1], objective_function(trajectory[-1]), marker='o', color='red', s=100)
    ax1.set_xlabel('$x_0$')
    ax1.set_ylabel('$x_1$')
    ax1.set_zlabel('$f(x)$')
    ax1.view_init(elev=20, azim=20)

    # Second 3D plot (contours and trajectory)
    ax2 = fig.add_subplot(1, 2, 2, projection='3d')
    
    # Reduce the contour levels to make the plot clearer
    ax2.contour(X, Y, Z, levels=np.linspace(Z.min(), Z.max(), 10), cmap='viridis')  # Sparse contour lines
    ax2.plot(X_vals, Y_vals, Z_vals, color='red', linewidth=3)  # Plot trajectory on contours
    ax2.scatter(trajectory[-1, 0], trajectory[-1, 1], objective_function(trajectory[-1]), marker='o', color='red', s=100)
    ax2.set_xlabel('$x_0$')
    ax2.set_ylabel('$x_1$')
    ax2.axes.zaxis.set_ticklabels([])  # Remove z axis ticks
    ax2.view_init(elev=90, azim=-90)

    plt.show()


Compute the minimum for different starting points and learning rates.

In the Ackley case, the trajectory may depend strongly on the initial point.  
This is a typical feature of non-convex optimization, where the algorithm may be attracted to different regions of the landscape.

In [ ]:
fminima, points, iters = steepest_descent_ls(
  ackley, ackley_gradient, x0 = np.array([-9, -9]), alpha=1, tolerance=1e-6)

In [ ]:
plot_optimization_trajectory_wider(points, iters, ackley)

### Griewank function

Repeat the process for the **Griewank function**:

\begin{equation}
f(x) = 1 + \frac{1}{4000} \left(x_1^2 + x_2^2\right) - \prod_{i=1}^{2} \cos\left(\frac{x_i}{\sqrt{i}}\right).
\end{equation}

Like the Ackley function, the Griewank function is non-convex and is commonly used to test optimization algorithms.  
It combines a quadratic growth term with oscillatory cosine terms, producing many local variations in the landscape.

In [ ]:
# Griewank function (2D)
def griewank(x):
    x1, x2 = x[0], x[1]
    term1 = (x1**2 + x2**2) / 4000
    term2 = np.cos(x1 / np.sqrt(1)) * np.cos(x2 / np.sqrt(2))
    return 1 + term1 - term2

# Gradient of the Griewank function (2D)
def griewank_gradient(x):
    x1, x2 = x[0], x[1]
    
    # Derivative with respect to x1
    df_dx1 = x1 / 2000 + np.sin(x1 / np.sqrt(1)) / np.sqrt(1)
    
    # Derivative with respect to x2
    df_dx2 = x2 / 2000 + np.sin(x2 / np.sqrt(2)) / np.sqrt(2)
    
    return np.array([df_dx1, df_dx2])

Plot the shape of the Griewank function by evaluating the following code.

In [ ]:
from itertools import product  # Import the product function from itertools

# Define the Griewank function (make sure it's defined or replace with the correct one)
#def Griewank_plot(x):
    # Example implementation of Griewank function
#    return 1 + (1/4000) * (x[0]**2 + x[1]**2) - np.cos(x[0]) * np.cos(x[1] / np.sqrt(2))

# Generate grid
x = np.arange(-5, 5, 0.025)
y = np.arange(-5, 5, 0.025)
X, Y = np.meshgrid(x, y)
Z = np.zeros(X.shape)

# Iterate through the grid to compute Z values using the Griewank function
mesh_size = range(len(X))
for i, j in product(mesh_size, mesh_size):
    x_coor = X[i][j]
    y_coor = Y[i][j]
    Z[i][j] = griewank(np.array([x_coor, y_coor]))

# Plot the surface
fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection='3d')  # Use add_subplot with 'projection' argument
ax.set_title('2D Griewank Function')
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_zlabel('$f(x_1, x_2)$')
ax.plot_surface(X, Y, Z, cmap='viridis')
plt.tight_layout()
plt.show()


Compute the minimum for different starting points and learning rates.

Compare the observed behaviour with the quadratic and Ackley examples:
- the quadratic function has a single, well-structured minimum;
- the Ackley and Griewank functions exhibit more intricate landscapes;
- adaptive step-size rules become especially useful when the geometry is less regular.

In [ ]:
fvalmin, xpoints, xiters = steepest_descent_ls(
  griewank, griewank_gradient, x0 = np.array([-2,9]), alpha=1, tolerance=1e-6)

Plot the resulting trajectory and discuss whether the iterates approach a local or global minimizer.

In [ ]:
plot_optimization_trajectory_wider(xpoints, xiters, griewank)